# Import danych: 9 plików CSV do bazy SQLite

Surowe dane Olist to 9 osobnych plików CSV. Dalsza analiza opiera się na łączeniu
tych tabel (zamówienia ↔ pozycje ↔ sprzedawcy ↔ recenzje), dlatego pierwszym krokiem
jest przeniesienie ich do jednej bazy SQLite — jedno źródło prawdy dla SQL, pandas
i Power BI.

**Wejście:** `data/raw/` — 9 plików CSV z Kaggle
**Wyjście:** `data/olist.db` — 9 tabel gotowych do zapytań

> Uwaga: ten notebook wgrywa dane *surowe*. Czyszczenie (typy dat, literówki,
> agregacja geolokalizacji) odbywa się w `02_data_cleaning.ipynb`, który nadpisuje
> wybrane tabele wersjami oczyszczonymi.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
BASE_DIR = Path("..")   # notebook uruchamiany z katalogu notebooks/
RAW_DIR  = BASE_DIR / "data" / "raw"
DB_PATH  = BASE_DIR / "data" / "olist.db"

## Mapowanie plików na tabele

Pliki źródłowe mają długie nazwy (`olist_customers_dataset.csv`), które w zapytaniach
byłyby nieczytelne. Słownik skraca je do krótkich nazw tabel (`customers`) i jest
jednym miejscem, w którym trzeba coś zmienić, gdy dojdzie nowy plik.

In [ ]:
tables = {
    "customers" : "olist_customers_dataset.csv",
    "orders" : "olist_orders_dataset.csv",
    "order_items" : "olist_order_items_dataset.csv",
    "payments" : "olist_order_payments_dataset.csv",
    "reviews" : "olist_order_reviews_dataset.csv",
    "products" : "olist_products_dataset.csv",
    "sellers" : "olist_sellers_dataset.csv",
    "geolocation" : "olist_geolocation_dataset.csv",
    "translations" : "product_category_name_translation.csv"
}

## Import do SQLite

Pętla wczytuje każdy plik i zapisuje go jako tabelę. `if_exists='replace'` sprawia,
że notebook jest idempotentny — można go uruchomić ponownie i baza zostanie zbudowana
od zera, bez duplikowania wierszy. `index=False` zapobiega zapisaniu indeksu pandas
jako dodatkowej kolumny.

In [4]:
conn = sqlite3.connect(DB_PATH)
for table_name, csv_file in tables.items():
    df = pd.read_csv(RAW_DIR / csv_file)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f'{table_name}: {len(df)} wierszy')
conn.close()

customers: 99441 wierszy


orders: 99441 wierszy


order_items: 112650 wierszy
payments: 103886 wierszy


reviews: 99224 wierszy
products: 32951 wierszy
sellers: 3095 wierszy


geolocation: 1000163 wierszy
translations: 71 wierszy


## Weryfikacja importu

Sprawdzenie, że wszystkie 9 tabel trafiło do bazy.

In [ ]:
conn = sqlite3.connect(DB_PATH)
tables_in_db = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables_in_db)
conn.close()

           name
0     customers
1        orders
2   order_items
3      payments
4       reviews
5      products
6       sellers
7   geolocation
8  translations
